## Cell 1 — Imports & Configuration

Load all required libraries and define global constants.
- `chess` / `chess.pgn` — parse PGN and represent board state
- `chess.engine` — UCI bridge to the Stockfish binary
- `DEPTH` — search depth used for every evaluation
- `THRESHOLDS` — centipawn-loss boundaries that map to move quality labels

In [3]:
import asyncio
import chess
import chess.engine
import chess.pgn
import io

# python-chess spawns Stockfish via asyncio subprocesses inside a background
# thread. On Windows, Jupyter uses SelectorEventLoop which does not support
# subprocess_exec — ProactorEventLoop does. Set the policy before any engine call.
if hasattr(asyncio, "WindowsProactorEventLoopPolicy"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ── Configuration ─────────────────────────────────────────────────────────────
STOCKFISH_PATH: str = r"C:\Users\Kenji\stockfish\stockfish-windows-x86-64-avx2.exe"
DEPTH: int = 15

# Centipawn-loss upper bounds per classification.
# Any loss above the "mistake" threshold is a blunder.
THRESHOLDS: dict[str, int] = {
    "best": 10,
    "good": 25,
    "inaccuracy": 50,
    "mistake": 100,
}

## Cell 2 — Sanity Check

Open the engine, evaluate the starting position at depth 10, print the score, then
close the engine. Run this cell once to confirm the Stockfish binary is reachable
before running the full analysis.

In [4]:
try:
    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    board = chess.Board()
    info = engine.analyse(board, chess.engine.Limit(depth=10))
    score = info["score"]
    print(f"Stockfish reachable. Starting position score: {score}")
    engine.quit()
except FileNotFoundError:
    print(f"ERROR: Stockfish binary not found at:\n  {STOCKFISH_PATH}")
    print("Update STOCKFISH_PATH in Cell 1 and re-run.")

Stockfish reachable. Starting position score: PovScore(Cp(+69), WHITE)


## Cell 3 — `analyze_game()` Function

Parses a PGN string, then walks every move while asking Stockfish to evaluate the
position before and after each ply. Centipawn loss is computed from the moving
player's perspective (positive = worse for them). The loss is mapped to a quality
label using `THRESHOLDS` and the result is collected into a list of dicts.

The engine is always closed via a `try/finally` block even if an error occurs.

In [5]:
def classify_move(cp_loss: int) -> str:
    # Walk thresholds in order; anything above "mistake" is a blunder.
    if cp_loss <= THRESHOLDS["best"]:
        return "best"
    if cp_loss <= THRESHOLDS["good"]:
        return "good"
    if cp_loss <= THRESHOLDS["inaccuracy"]:
        return "inaccuracy"
    if cp_loss <= THRESHOLDS["mistake"]:
        return "mistake"
    return "blunder"


def analyze_game(pgn: str) -> list[dict]:
    """
    Analyse every move in a PGN string with Stockfish.

    Returns a list of dicts, one per move:
      move_number   : int   — full move number (1-based)
      color         : str   — "white" or "black"
      move_san      : str   — move in Standard Algebraic Notation
      cp_loss       : int   — centipawn loss (0 = perfect, higher = worse)
      classification: str   — best / good / inaccuracy / mistake / blunder
    """
    game = chess.pgn.read_game(io.StringIO(pgn))

    if game is None:
        raise ValueError("Could not parse PGN string — check that it is valid.")

    results: list[dict] = []
    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)

    try:
        board = game.board()

        for node in game.mainline():
            move = node.move
            color = "white" if board.turn == chess.WHITE else "black"
            move_number = board.fullmove_number
            move_san = board.san(move)

            # Evaluate position BEFORE the move (from mover's perspective)
            info_before = engine.analyse(board, chess.engine.Limit(depth=DEPTH))
            score_before = info_before["score"].relative.score(mate_score=10000)

            # Apply the move
            board.push(move)

            # Evaluate position AFTER the move.
            # .relative now gives the score from the *opponent's* POV,
            # so negate it to keep it from the mover's POV.
            info_after = engine.analyse(board, chess.engine.Limit(depth=DEPTH))
            score_after = -info_after["score"].relative.score(mate_score=10000)

            # Centipawn loss: how much worse the position became for the mover.
            # Clamp at 0 so a move that happens to gain does not go negative.
            cp_loss = max(0, score_before - score_after)

            results.append({
                "move_number": move_number,
                "color": color,
                "move_san": move_san,
                "cp_loss": cp_loss,
                "classification": classify_move(cp_loss),
            })
    finally:
        engine.quit()

    return results

## Cell 4 — Test with Sample PGN

Run `analyze_game()` against a known game so we can inspect its output before
wiring it up to live API data. The PGN is embedded as a raw string literal to
avoid any encoding issues.

In [10]:
SAMPLE_PGN: str = r"""
[Event "Live Chess"]
[White "OrangeMutante"]
[Black "magickids12"]
[Result "1-0"]
[TimeControl "60"]

1. e4 e5 2. Nf3 Nc6 3. c3 Nf6 4. d4 exd4 5. e5 Qe7 6. cxd4 Ng4 7. h3 Nh6
8. Bxh6 gxh6 9. Be2 Bg7 10. O-O O-O 11. Nc3 Qb4 12. Qd2 Re8 13. a3 Qb5
14. Bxb5 a6 15. Bxc6 bxc6 16. b4 d5 17. exd6 cxd6 18. Rfe1 Be6 19. d5 cxd5
20. Nxd5 Bxd5 21. Rxe8+ Rxe8 22. Qxd5 Rd8 23. Re1 Be5 24. Qc6 Rf8 25. Nxe5
dxe5 26. Rxe5 f6 27. Re7 Rf7 28. Qc7 Rxe7 29. Qxe7 Kh8 30. Qxf6+ Kg8
31. Qe7 Kh8 32. Qa7 Kg8 33. Qxa6 h5 34. Qa7 h4 35. Qc7 h6 36. b5 h5 37. b6 1-0
"""

print("Running analysis — this may take a minute at depth 15...")
move_data: list[dict] = analyze_game(SAMPLE_PGN)
print(f"Done. {len(move_data)} moves analysed.")

Running analysis — this may take a minute at depth 15...
Done. 73 moves analysed.


## Cell 5 — Display Results

Print a formatted move table followed by a per-player summary that includes
classification counts and an accuracy percentage.

Accuracy formula: `max(0, min(100, 100 - avg_cp_loss / 10))`

In [11]:
import math

# ── Move table ────────────────────────────────────────────────────────────────
print(f"{'Move':<6} {'Color':<6} {'SAN':<8} {'CP Loss':>8}  Classification")
print("-" * 50)

for entry in move_data:
    label = entry["classification"].upper()
    print(
        f"Move {entry['move_number']:<2} "
        f"{entry['color']:<6} "
        f"{entry['move_san']:<8} "
        f"cp_loss: {entry['cp_loss']:>4}  "
        f"{label}"
    )

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print(f"Total moves analysed: {len(move_data)}")
print()

all_classifications: list[str] = ["best", "good", "inaccuracy", "mistake", "blunder"]

for color in ("white", "black"):
    color_moves = [m for m in move_data if m["color"] == color]

    if not color_moves:
        continue

    total_cp_loss = sum(m["cp_loss"] for m in color_moves)
    avg_cp_loss = total_cp_loss / len(color_moves)
    accuracy = max(0, min(100, math.floor(100 - avg_cp_loss / 10)))

    print(f"{color.capitalize()} ({len(color_moves)} moves):")

    for classification in all_classifications:
        count = sum(1 for m in color_moves if m["classification"] == classification)
        print(f"  {classification:<12}: {count}")

    print(f"  avg cp loss : {avg_cp_loss:.1f}")
    print(f"  accuracy    : {accuracy}%")
    print()

Move   Color  SAN       CP Loss  Classification
--------------------------------------------------
Move 1  white  e4       cp_loss:    3  BEST
Move 1  black  e5       cp_loss:    0  BEST
Move 2  white  Nf3      cp_loss:    0  BEST
Move 2  black  Nc6      cp_loss:   11  GOOD
Move 3  white  c3       cp_loss:   53  MISTAKE
Move 3  black  Nf6      cp_loss:   13  GOOD
Move 4  white  d4       cp_loss:    0  BEST
Move 4  black  exd4     cp_loss:   14  GOOD
Move 5  white  e5       cp_loss:    6  BEST
Move 5  black  Qe7      cp_loss:   73  MISTAKE
Move 6  white  cxd4     cp_loss:    4  BEST
Move 6  black  Ng4      cp_loss:  195  BLUNDER
Move 7  white  h3       cp_loss:    0  BEST
Move 7  black  Nh6      cp_loss:    0  BEST
Move 8  white  Bxh6     cp_loss:  114  BLUNDER
Move 8  black  gxh6     cp_loss:    0  BEST
Move 9  white  Be2      cp_loss:   72  MISTAKE
Move 9  black  Bg7      cp_loss:   17  GOOD
Move 10 white  O-O      cp_loss:    0  BEST
Move 10 black  O-O      cp_loss:    0  BEST
Move 1

## Interactive Game Navigator

Step through the game move-by-move with a live SVG chess board.
Use **◀ Prev** and **Next ▶** to navigate positions.
Move quality is colour-coded below the board (green = best, red = blunder),
and a centipawn-loss eval bar shows which side benefited from each move.

In [19]:
import chess.svg
import ipywidgets as widgets
from IPython.display import display


def build_navigator(move_data: list[dict]) -> None:
    """
    Render an interactive move-by-move chess board navigator using ipywidgets.

    Board snapshots are pre-computed once from SAMPLE_PGN so that button
    clicks never re-invoke the engine — only stored move_data is used.

    Parameters
    ----------
    move_data : list[dict]
        Output of analyze_game(): each entry has move_number, color,
        move_san, cp_loss, and classification.
    """

    # ── Pre-compute FEN snapshots and Move objects ─────────────────────────
    # snapshots[i] = FEN after i moves (index 0 = starting position).
    # moves[i]     = chess.Move played at step i+1 (used for lastmove highlight).
    game = chess.pgn.read_game(io.StringIO(SAMPLE_PGN))
    replay_board = game.board()
    snapshots: list[str] = [replay_board.fen()]
    moves: list[chess.Move] = []

    for node in game.mainline():
        moves.append(node.move)
        replay_board.push(node.move)
        snapshots.append(replay_board.fen())

    total_moves: int = len(move_data)

    # ── Classification → HTML colour map ──────────────────────────────────
    classification_colors: dict[str, str] = {
        "best": "#27ae60",
        "good": "#16a085",
        "inaccuracy": "#f39c12",
        "mistake": "#e67e22",
        "blunder": "#e74c3c",
    }

    # ── Mutable navigation state ───────────────────────────────────────────
    # Stored in a dict so inner callbacks can mutate it without nonlocal.
    state: dict[str, int] = {"index": 0}

    # ── Widgets ────────────────────────────────────────────────────────────
    # widgets.HTML renders .value as live HTML/SVG — no Output wrapper needed.
    board_widget: widgets.HTML = widgets.HTML()
    info_widget: widgets.HTML = widgets.HTML()

    prev_btn: widgets.Button = widgets.Button(
        description="◀ Prev",
        disabled=True,
        layout=widgets.Layout(width="100px"),
    )
    next_btn: widgets.Button = widgets.Button(
        description="Next ▶",
        disabled=(total_moves == 0),
        layout=widgets.Layout(width="100px"),
    )
    move_label: widgets.Label = widgets.Label(
        value=f"Move 0 / {total_moves}",
        layout=widgets.Layout(width="150px", margin="0 8px"),
    )

    # ── Helper: build a text eval bar from cp_loss ─────────────────────────
    def build_eval_bar(cp_loss: int, color: str) -> str:
        # Cap at 500 cp (5 pawns) so the bar does not saturate on large blunders.
        capped: int = min(500, cp_loss)
        filled: int = int((capped / 500) * 10)
        empty: int = 10 - filled
        pawns: float = capped / 100.0

        if color == "white":
            # White lost cp → black now benefits → negative score, blocks on right.
            score_str: str = f"-{pawns:.2f}"
            bar: str = "░" * empty + "█" * filled
        else:
            # Black lost cp → white now benefits → positive score, blocks on left.
            score_str = f"+{pawns:.2f}"
            bar = "█" * filled + "░" * empty

        return f"{score_str} {bar}"

    # ── Helper: refresh all widgets to reflect the given board index ───────
    def render(index: int) -> None:
        # Reconstruct board object from pre-computed FEN.
        snapshot_board: chess.Board = chess.Board(snapshots[index])

        # Resolve the last move for highlighting; None at the starting position.
        last_move: chess.Move | None = moves[index - 1] if index > 0 else None

        svg_str: str = chess.svg.board(
            snapshot_board,
            size=400,
            lastmove=last_move,
            colors={
                "square light": "#f0d9b5",
                "square dark": "#b58863",
                "lastmove": "#cdd26a",
            },
        )

        # Assign directly to .value — widgets.HTML re-renders SVG immediately.
        board_widget.value = svg_str

        if index == 0:
            info_widget.value = "<em>Starting position — press Next ▶ to begin</em>"
        else:
            entry: dict = move_data[index - 1]
            cls_color: str = classification_colors.get(
                entry["classification"], "#000000"
            )
            eval_bar: str = build_eval_bar(entry["cp_loss"], entry["color"])
            info_widget.value = (
                f"<b>Move {entry['move_number']} "
                f"({entry['color'].capitalize()}): "
                f"{entry['move_san']}</b>"
                f"&nbsp;&nbsp;"
                f"<span style='color:{cls_color}'>"
                f"<b>{entry['classification'].upper()}</b>"
                f"</span>"
                f"&nbsp; cp_loss: <b>{entry['cp_loss']}</b>"
                f"&nbsp;&nbsp;<code>{eval_bar}</code>"
            )

        move_label.value = f"Move {index} / {total_moves}"
        prev_btn.disabled = (index == 0)
        next_btn.disabled = (index == total_moves)

    # ── Button click handlers ──────────────────────────────────────────────
    def on_prev_click(btn: widgets.Button) -> None:
        if state["index"] > 0:
            state["index"] -= 1
            render(state["index"])

    def on_next_click(btn: widgets.Button) -> None:
        if state["index"] < total_moves:
            state["index"] += 1
            render(state["index"])

    prev_btn.on_click(on_prev_click)
    next_btn.on_click(on_next_click)

    # Initial render at the starting position (index 0).
    render(0)

    # ── Assemble final layout ──────────────────────────────────────────────
    button_row: widgets.HBox = widgets.HBox(
        [prev_btn, move_label, next_btn],
        layout=widgets.Layout(align_items="center", margin="6px 0"),
    )
    navigator: widgets.VBox = widgets.VBox([board_widget, button_row, info_widget])
    display(navigator)

In [20]:
# Display the interactive navigator using move_data already computed in Cell 4.
# No analysis is re-run here.
build_navigator(move_data)